# Problema de Corte Unidimensional

**Trabalho AV2 — Modelagem em Programação Matemática (Unifor)**

## Enunciado

Dada uma barra de comprimento $L$ e um conjunto de itens com tamanhos $\ell_1,\ell_2,\ldots,\ell_n$ e demandas $d_1,d_2,\ldots,d_n$, determinar como cortar as barras de modo a **satisfazer toda a demanda minimizando o desperdício total** (sobras).

*Caso do enunciado:* $L=150$, itens $\ell=(80,60,50)$, demanda $d=(70,100,120)$.

---

## Formulação — PLI baseada em padrões de corte (Gilmore-Gomory)

Seja $P$ o conjunto de **padrões de corte maximais válidos**. Cada padrão $j\in P$ é um vetor $(a_{1j},a_{2j},\ldots,a_{nj})$ com $a_{ij}\in\mathbb{Z}_{\ge 0}$ representando quantos itens de cada tipo são cortados de uma barra usando aquele padrão. Validade e maximalidade:

$$\sum_{i=1}^{n} a_{ij}\,\ell_i \;\le\; L \qquad\text{e}\qquad L-\sum_{i=1}^{n} a_{ij}\,\ell_i \;<\; \min_i \ell_i.$$

O **desperdício** do padrão $j$ é $w_j = L - \sum_{i=1}^{n} a_{ij}\,\ell_i$.

**Variáveis de decisão.** $x_j \in \mathbb{Z}_{\ge 0}$: número de barras cortadas usando o padrão $j$.

**Função objetivo.**
$$\min \;\; \sum_{j\in P} w_j\, x_j$$

**Restrições de demanda (igualdade — atendimento exato).**
$$\sum_{j\in P} a_{ij}\, x_j \;=\; d_i, \qquad \forall\, i=1,\ldots,n$$

**Domínio.** $x_j \in \mathbb{Z}_{\ge 0},\ \forall j\in P$.

In [1]:
!pip install ortools -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Program Files\Python310\python.exe -m pip install --upgrade pip


## 1. Entrada de dados

In [2]:
# Leitura interativa dos dados de entrada
L = int(input('Tamanho da barra: '))
n = int(input('Quantidade de tipos de itens: '))

l = []
for i in range(n):
    l.append(int(input(f'Tamanho do item tipo {i+1}: ')))

d = []
for i in range(n):
    d.append(int(input(f'Demanda do item tipo {i+1}: ')))

print()
print(f'Barra: {L}')
print(f'Tipos de itens: {n}')
print(f'Tamanhos: {l}')
print(f'Demandas: {d}')


Barra: 150
Tipos de itens: 3
Tamanhos: [80, 60, 50]
Demandas: [70, 100, 120]


## 2. Geração dos padrões de corte maximais

Enumeramos, recursivamente, todas as combinações $(a_1,\ldots,a_n)$ tais que $\sum a_i\ell_i \le L$ e que sejam **maximais**: a sobra é menor que o menor item, de modo que nenhum item adicional caberia.

In [3]:
def gerar_padroes(L, l):
    n = len(l)
    menor_item = min(l)
    padroes = []
    a = [0] * n

    def busca(i, restante):
        if i == n:
            # padrao eh maximal se nao cabe mais nenhum item na sobra
            if restante < menor_item:
                padroes.append(tuple(a))
            return
        max_qtd = restante // l[i]
        for q in range(max_qtd + 1):
            a[i] = q
            busca(i + 1, restante - q * l[i])
        a[i] = 0

    busca(0, L)
    return padroes


padroes = gerar_padroes(L, l)

print(f'Total de padroes maximais gerados: {len(padroes)}')
print()
print('Padroes de corte:')
cabecalho = '  j  | ' + ' | '.join(f'a_{i+1}(={l[i]})' for i in range(n)) + ' | usado | desperdicio'
print(cabecalho)
print('-' * len(cabecalho))
for j, p in enumerate(padroes):
    usado = sum(p[i] * l[i] for i in range(n))
    desp = L - usado
    coefs = ' | '.join(f'{p[i]:>6d}' for i in range(n))
    print(f' {j+1:>3d} | {coefs} | {usado:>5d} | {desp:>11d}')

Total de padroes maximais gerados: 5

Padroes de corte:
  j  | a_1(=80) | a_2(=60) | a_3(=50) | usado | desperdicio
-----------------------------------------------------------
   1 |      0 |      0 |      3 |   150 |           0
   2 |      0 |      1 |      1 |   110 |          40
   3 |      0 |      2 |      0 |   120 |          30
   4 |      1 |      0 |      1 |   130 |          20
   5 |      1 |      1 |      0 |   140 |          10


## 3. Construção do modelo no OR-Tools

In [4]:
from ortools.linear_solver import pywraplp

solver = pywraplp.Solver.CreateSolver('SCIP')
if solver is None:
    raise RuntimeError('Nao foi possivel criar o solver SCIP.')

# Variaveis: x[j] = numero de barras cortadas segundo o padrao j (inteira >= 0)
x = {j: solver.IntVar(0, solver.infinity(), f'x{j+1}') for j in range(len(padroes))}

In [5]:
# Funcao objetivo: minimizar o desperdicio total
objetivo = solver.Objective()
for j, p in enumerate(padroes):
    desperdicio_j = L - sum(p[i] * l[i] for i in range(n))
    objetivo.SetCoefficient(x[j], desperdicio_j)
objetivo.SetMinimization()

In [6]:
# Restricoes de demanda: para cada tipo i, a producao deve atender exatamente d[i]
for i in range(n):
    ct = solver.RowConstraint(d[i], d[i], f'demanda_item{i+1}')
    for j, p in enumerate(padroes):
        if p[i] > 0:
            ct.SetCoefficient(x[j], p[i])

## 4. Modelo de Programação Linear Inteira (formato LP)

In [7]:
print(solver.ExportModelAsLpFormat(False))

\ Generated by MPModelProtoExporter
\   Name             : 
\   Format           : Free
\   Constraints      : 3
\   Variables        : 5
\     Binary         : 0
\     Integer        : 5
\     Continuous     : 0
Minimize
 Obj: +40 x2 +30 x3 +20 x4 +10 x5 
Subject to
 demanda_item1: +1 x4 +1 x5  = 70
 demanda_item2: +1 x2 +2 x3 +1 x5  = 100
 demanda_item3: +3 x1 +1 x2 +1 x4  = 120
Bounds
 0 <= x1 <= inf
 0 <= x2 <= inf
 0 <= x3 <= inf
 0 <= x4 <= inf
 0 <= x5 <= inf
Generals
 x1
 x2
 x3
 x4
 x5
End



## 5. Resolução e solução ótima

In [8]:
status = solver.Solve()

if status == pywraplp.Solver.OPTIMAL:
    desperdicio_total = int(round(objetivo.Value()))
    usados = [(j, int(round(x[j].solution_value()))) for j in range(len(padroes)) if x[j].solution_value() > 0.5]

    total_barras = sum(q for _, q in usados)

    print('Solucao otima encontrada.')
    print()
    print('Padroes utilizados:')
    for j, q in usados:
        p = padroes[j]
        composicao = ', '.join(f'{p[i]}x{l[i]}' for i in range(n) if p[i] > 0)
        desp_j = L - sum(p[i] * l[i] for i in range(n))
        print(f'  x{j+1} = {q:>4d}  -> padrao ({composicao})   desperdicio/barra = {desp_j}')
    print()

    print('Producao vs. demanda:')
    for i in range(n):
        produzido = sum(q * padroes[j][i] for j, q in usados)
        print(f'  item {i+1} (tamanho {l[i]}): produzido = {produzido}, demanda = {d[i]}')
    print()

    print(f'Total de barras utilizadas: {total_barras}')
    print(f'Desperdicio total: {desperdicio_total}')
else:
    print('Modelo sem solucao otima.')

Solucao otima encontrada.

Padroes utilizados:
  x1 =   40  -> padrao (3x50)   desperdicio/barra = 0
  x3 =   15  -> padrao (2x60)   desperdicio/barra = 30
  x5 =   70  -> padrao (1x80, 1x60)   desperdicio/barra = 10

Producao vs. demanda:
  item 1 (tamanho 80): produzido = 70, demanda = 70
  item 2 (tamanho 60): produzido = 100, demanda = 100
  item 3 (tamanho 50): produzido = 120, demanda = 120

Total de barras utilizadas: 125
Desperdicio total: 1150
